# Van der Pol equation

In [ ]:
import numpy as np
from scipy.optimize import root
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"

We consider the following problem :

$$
\left\{ 
\begin{aligned} 
{\mathrm d}_t x & = \epsilon^2 \left(y-\frac{x^3}{3}+x \right)\\ 
{\mathrm d}_t y & = -x 
\end{aligned} 
\right. 
\qquad{} \text{with } \epsilon > 0
$$

In [ ]:
class vanderpol_model:

    def __init__(self, eps):
        self.eps = eps

    def fcn(self, t, y):
        y1, y2 = y
        eps = self.eps
        y1_dot = (eps**2)*(y2-y1**3/3+y1)
        y2_dot = -y1
        return np.array([y1_dot, y2_dot])

    def jac(self, t, y):
        y1, y2 = y
        eps = self.eps
        return np.array([[eps**2*(-y1**2 + 1), eps**2], [-1, 0]])

## Runge-Kutta vs Radau  

In order to compute the global error, the quasi-exact solution is obtained by using the RaduIIA  method of order 8 with stepsize control and fine tolerances.

In [ ]:
tini = 0. 
tend = 10.

yini = (0.5, 0)

eps = 40
vdpm = vanderpol_model(eps)
fcn = vdpm.fcn
jac = vdpm.jac

tol_rk = 1e-6
sol_rk = solve_ivp(fcn, (tini, tend), yini, method="RK45", rtol=tol_rk, atol=tol_rk)
sol_exa = solve_ivp(fcn, (tini, tend), yini, method="Radau", rtol=1e-12, atol=1e-12, t_eval=sol_rk.t, jac=jac)
eig_vals = np.zeros((sol_rk.t.size, 2), dtype=np.complex128)
for it in range(0,sol_rk.t.size):
    eig_vals[it] = np.linalg.eigvals(jac(0., sol_rk.y[:,it]))
lambda1 = eig_vals[:, 0]

err_rk = np.abs(sol_exa.y[0]-sol_rk.y[0])
print("Dopri5 integration :")
print("  Tolerance used                    = ", tol_rk)
print("  Number of accepted steps          = ", sol_rk.t.size-1)
print("  Number of function evaluations    = ", sol_rk.nfev)
print(f"  Error l2 norm (variable x)        = {np.linalg.norm(err_rk):e}")

tol_rad = 1e-6
sol_rad = solve_ivp(fcn, (tini, tend), yini, method="Radau", rtol=tol_rad, atol=tol_rad, jac=jac)
sol_exa = solve_ivp(fcn, (tini, tend), yini, method="Radau", rtol=1e-12, atol=1e-12, t_eval=sol_rad.t, jac=jac)
err_rad  = np.abs(sol_exa.y[0]-sol_rad.y[0])
print("\nRadauIIA integration :")
print("  Tolerance used                    = ", tol_rad)
print("  Number of accepted steps          = ", sol_rad.t.size-1)
print("  Number of function evaluations    = ", sol_rad.nfev)
print("  Number of jacobian evaluations    = ", sol_rad.njev)
print("  Number of lu decompositions       = ", sol_rad.nlu)
print(f"  Error l2 norm (variable x)        = {np.linalg.norm(err_rad):e}")

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.1, subplot_titles=("Solution x", "Error"))
fig.add_trace(go.Scatter(x=sol_rk.t, y=sol_rk.y[0], name=f'RK  (tol={tol_rk})', legendgroup='rk', mode='lines', marker_color='rgb(76,114,176)'), row=1, col=1)
fig.add_trace(go.Scatter(x=sol_rad.t, y=sol_rad.y[0], name=f'RadauIIA  (tol={tol_rad})', legendgroup='rad', mode='lines', marker_color='rgb(221,132,82)'), row=1, col=1)
fig.add_trace(go.Scatter(x=sol_rk.t, y=err_rk,  legendgroup='rk',  showlegend=False, mode='lines', marker_color='rgb(76,114,176)'), row=2, col=1)
fig.add_trace(go.Scatter(x=sol_rad.t, y=err_rad,  legendgroup='rad', showlegend=False, mode='lines', marker_color='rgb(221,132,82)'), row=2, col=1)
legend = dict(y=1.10, bgcolor='rgba(0,0,0,0)', orientation="h")
fig.update_layout(legend=legend, height=750)
fig.update_xaxes(title_text='t', row=2)
fig.update_yaxes(type="log", exponentformat='e', row=2)
fig.show()

In [ ]:
dt_rk = np.array(sol_rk.t[1:]) - np.array(sol_rk.t[:-1])
t_rk  = np.array(sol_rk.t[:-1]) + 0.5*dt_rk


dt_rad = np.array(sol_rad.t[1:]) - np.array(sol_rad.t[:-1])
t_rad  = np.array(sol_rad.t[:-1]) + 0.5*dt_rad

fig = make_subplots(rows=3, cols=1, vertical_spacing=0.1, subplot_titles=("Real part of maximum eigen value",  "Timestep DOPRI5", "Timestep RADAUIIA"))
fig.add_trace(go.Scatter(x=sol_rk.t, y=np.real(lambda1), showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=t_rk, y=dt_rk, width=dt_rk, showlegend=False, marker_color="rgb(85,168,104)", marker_line_width=0.05), row=2, col=1)
fig.add_trace(go.Bar(x=t_rad, y=dt_rad, width=dt_rad, showlegend=False), row=3, col=1)
fig.update_layout(height=1000)
fig.update_xaxes(title_text='t', row=3)
fig.show()